# 🧬 GNN Drug Discovery — Complete Walkthrough

This notebook covers the **full pipeline** end-to-end:

1. [Setup & Imports](#1-setup)
2. [Data Loading — ogbg-molhiv](#2-data-loading)
3. [Data Exploration](#3-data-exploration)
4. [Molecule → Graph (SMILES to PyG)](#4-smiles-to-graph)
5. [Model Architecture (GIN & GCN)](#5-model-architecture)
6. [Training](#6-training)
7. [Evaluation & Results](#7-evaluation)
8. [Single Molecule Prediction](#8-single-prediction)
9. [Tox21 Virtual Screening](#9-tox21-screening)
10. [Key Takeaways](#10-takeaways)

---
**Dataset:** ogbg-molhiv (41,127 molecules, NCI HIV inhibition screen)  
**Model:** Graph Isomorphism Network (GIN) with pos_weight for class imbalance  
**Metric:** ROC-AUC (scaffold split)  
**Runtime:** ~15 min on CPU · ~3 min on GPU


## 1. Setup

Install dependencies if running outside the repo environment.

In [ ]:
# Uncomment and run if needed (e.g. on Colab)
# !pip install torch torch_geometric ogb rdkit-pypi pandas numpy scikit-learn matplotlib tqdm

In [ ]:
from __future__ import annotations

import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from torch import nn
from torch_geometric.data import Batch, Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINConv, GCNConv, global_add_pool, global_mean_pool

from ogb.graphproppred import PygGraphPropPredDataset
from ogb.utils import smiles2graph
from sklearn.metrics import roc_auc_score, average_precision_score, RocCurveDisplay

warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')

In [ ]:
# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

## 2. Data Loading — ogbg-molhiv

The **ogbg-molhiv** dataset comes from the NCI DTP AIDS Antiviral Screen, which tested ~40,000 compounds for HIV inhibition in human T-cells.

- **Label = 1 (CM):** Confirmed active — molecule reduced HIV replication without harming host cells
- **Label = 0 (CI/CA):** Inactive or moderately active — no useful antiviral effect
- **Split:** Scaffold (Bemis-Murcko) — molecules grouped by core ring structure, test set has unseen scaffolds

> ⚠️ **Phenotypic data:** Labels reflect a *cell-based outcome* — not which HIV protein is targeted or the mechanism. The model learns "does this molecular structure correlate with anti-HIV activity" — not *why* it works.

In [ ]:
print('Loading ogbg-molhiv...')
dataset = PygGraphPropPredDataset(name='ogbg-molhiv')
split_idx = dataset.get_idx_split()

train_data = dataset[split_idx['train']]
valid_data = dataset[split_idx['valid']]
test_data  = dataset[split_idx['test']]

print(f'Total molecules : {len(dataset):,}')
print(f'Train           : {len(train_data):,}')
print(f'Validation      : {len(valid_data):,}')
print(f'Test            : {len(test_data):,}')
print(f'Node features   : {dataset.num_node_features}')
print(f'Edge features   : {dataset.num_edge_features}')
print(f'Task type       : {dataset.task_type}')

In [ ]:
BATCH_SIZE = 64
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_data,  batch_size=BATCH_SIZE, shuffle=False)
print(f'Batches — train: {len(train_loader)}, val: {len(valid_loader)}, test: {len(test_loader)}')

## 3. Data Exploration

In [ ]:
# Class balance
all_labels = torch.cat([data.y for data in dataset]).view(-1).numpy()
n_pos = int((all_labels == 1).sum())
n_neg = int((all_labels == 0).sum())
print(f'HIV Active (label=1): {n_pos:,}  ({100*n_pos/len(all_labels):.1f}%)')
print(f'HIV Inactive (label=0): {n_neg:,}  ({100*n_neg/len(all_labels):.1f}%)')

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Class distribution
axes[0].bar(['Inactive', 'Active'], [n_neg, n_pos], color=['#6b7280', '#ef4444'])
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate([n_neg, n_pos]):
    axes[0].text(i, v + 100, f'{v:,}', ha='center', fontweight='bold')

# Node count distribution
sample = [dataset[i] for i in range(min(2000, len(dataset)))]
node_counts = [d.num_nodes for d in sample]
axes[1].hist(node_counts, bins=40, color='#3b82f6', edgecolor='white')
axes[1].set_title('Atoms per Molecule')
axes[1].set_xlabel('Atom count')
axes[1].set_ylabel('Frequency')
axes[1].axvline(np.mean(node_counts), color='red', linestyle='--', label=f'Mean={np.mean(node_counts):.1f}')
axes[1].legend()

# Edge count distribution
edge_counts = [d.edge_index.shape[1] for d in sample]
axes[2].hist(edge_counts, bins=40, color='#10b981', edgecolor='white')
axes[2].set_title('Bonds per Molecule')
axes[2].set_xlabel('Bond count (edges)')
axes[2].set_ylabel('Frequency')
axes[2].axvline(np.mean(edge_counts), color='red', linestyle='--', label=f'Mean={np.mean(edge_counts):.1f}')
axes[2].legend()

plt.suptitle('ogbg-molhiv Dataset Exploration', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Inspect a single graph
example = dataset[0]
print('Single molecule graph:')
print(f'  x (node features) : {example.x.shape}  — {example.x.shape[0]} atoms × {example.x.shape[1]} features')
print(f'  edge_index         : {example.edge_index.shape}  — 2 × {example.edge_index.shape[1]} directed bonds')
print(f'  edge_attr          : {example.edge_attr.shape}')
print(f'  y (label)          : {example.y.item()}  ({"active" if example.y.item()==1 else "inactive"})')
print(f'\nFirst 3 atom feature vectors:')
print(example.x[:3])
print('\nNode feature columns: [atomic_num, chirality, degree, formal_charge, num_H, H_bonding, aromaticity, ...]')

## 4. Molecule → Graph (SMILES to PyG)

Any molecule described as a SMILES string can be converted to a PyG `Data` object for direct inference.

In [ ]:
def smiles_to_pyg(smiles: str) -> Data:
    """Convert a SMILES string to a PyTorch Geometric Data object."""
    graph = smiles2graph(smiles)
    return Data(
        x=torch.from_numpy(graph['node_feat']).to(torch.long),
        edge_index=torch.from_numpy(graph['edge_index']).to(torch.long),
        edge_attr=torch.from_numpy(graph['edge_feat']).to(torch.long),
    )

# Example molecules
examples = {
    'Aspirin':         'CC(=O)OC1=CC=CC=C1C(=O)O',
    'AZT (HIV drug)':  'CC1=CN(C(=O)NC1=O)[C@@H]2C[C@@H](N=[N+]=[N-])[C@H](CO)O2',
    'Caffeine':        'CN1C=NC2=C1C(=O)N(C(=O)N2C)C',
    'Efavirenz (HIV)': 'FC(F)(F)C1(CC#C)OC2=NC(=NC2=C1)Cl',  # NNRTI
}

for name, smi in examples.items():
    g = smiles_to_pyg(smi)
    print(f'{name:25s}: {g.num_nodes:3d} atoms, {g.edge_index.shape[1]:3d} bond-edges')

## 5. Model Architecture

### Why GIN over GCN?

| | GCN | GIN |
|---|---|---|  
| Aggregation | Mean | Sum |
| Pooling | global_mean_pool | global_add_pool |
| Expressiveness | Below 1-WL test | Exactly 1-WL (max for MPNNs) |
| Why sum? | Mean loses count info: 2 neighbours w/ [1,1] = 1 neighbour w/ [2] | Sum distinguishes these |
| Reference | Kipf & Welling, ICLR 2017 | Xu et al., ICLR 2019 |

In [ ]:
class GINClassifier(nn.Module):
    """
    Graph Isomorphism Network for binary graph classification.
    
    Architecture:
        - 4 GINConv layers with MLP aggregators
        - Global sum pooling
        - 2-layer MLP classification head
    
    Reference: Xu et al., "How Powerful are Graph Neural Networks?" ICLR 2019
    """
    def __init__(self, num_node_features: int, hidden_dim: int = 128,
                 num_layers: int = 4, dropout: float = 0.2):
        super().__init__()
        self.dropout = dropout
        self.convs = nn.ModuleList()
        
        # First layer: node_features -> hidden_dim
        mlp0 = nn.Sequential(
            nn.Linear(num_node_features, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        self.convs.append(GINConv(mlp0))
        
        # Remaining layers: hidden_dim -> hidden_dim
        for _ in range(num_layers - 1):
            mlp = nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim)
            )
            self.convs.append(GINConv(mlp))
        
        # Classification head
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x, edge_index, batch, return_embeddings=False):
        x = x.float()
        for conv in self.convs:
            x = F.relu(conv(x, edge_index))
            x = F.dropout(x, p=self.dropout, training=self.training)
        node_emb = x  # per-atom embeddings [N, hidden_dim]
        graph_emb = global_add_pool(x, batch)  # [num_graphs, hidden_dim]
        out = self.head(graph_emb).view(-1)
        if return_embeddings:
            return out, node_emb
        return out


class GCNClassifier(nn.Module):
    """GCN baseline — mean aggregation, global mean pool. Kipf & Welling ICLR 2017."""
    def __init__(self, num_node_features: int, hidden_dim: int = 128,
                 num_layers: int = 4, dropout: float = 0.2):
        super().__init__()
        self.dropout = dropout
        self.convs = nn.ModuleList()
        self.convs.append(GCNConv(num_node_features, hidden_dim))
        for _ in range(num_layers - 1):
            self.convs.append(GCNConv(hidden_dim, hidden_dim))
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x, edge_index, batch, return_embeddings=False):
        x = x.float()
        for conv in self.convs:
            x = F.relu(conv(x, edge_index))
            x = F.dropout(x, p=self.dropout, training=self.training)
        node_emb = x
        graph_emb = global_mean_pool(x, batch)
        out = self.head(graph_emb).view(-1)
        if return_embeddings:
            return out, node_emb
        return out


# Instantiate and inspect
NUM_FEATURES = dataset.num_node_features
HIDDEN_DIM   = 128
NUM_LAYERS   = 4
DROPOUT      = 0.2

model = GINClassifier(NUM_FEATURES, HIDDEN_DIM, NUM_LAYERS, DROPOUT).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f'\nTrainable parameters: {n_params:,}')

## 5b. GINE — GIN with Edge Features

`GINEConv` extends GIN by incorporating `edge_attr` (bond type, stereochemistry, ring membership) into each message-passing step. These features are currently **ignored** by plain GIN — GINE fixes that.

| | GIN | GINE |
|---|---|---|
| Edge features | ❌ Ignored | ✅ Used (via GINEConv) |
| Aggregation | Sum | Sum |
| Expected AUC gain | baseline | +1–2% |
| Reference | Xu et al. ICLR 2019 | Hu et al. ICLR 2020 |

In [ ]:
from torch_geometric.nn import GINEConv

class GINEClassifier(nn.Module):
    """
    Graph Isomorphism Network with Edge features (GINE).
    Uses GINEConv — extends GIN by passing edge_attr through each message step.
    Bond type, stereochemistry, and ring membership are all used.
    Reference: Hu et al., 'Strategies for Pre-training GNNs', ICLR 2020
    """
    def __init__(self, num_node_features, num_edge_features=3,
                 hidden_dim=128, num_layers=4, dropout=0.2):
        super().__init__()
        self.dropout = dropout
        self.convs = nn.ModuleList()
        mlp0 = nn.Sequential(
            nn.Linear(num_node_features, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        self.convs.append(GINEConv(mlp0, edge_dim=num_edge_features))
        for _ in range(num_layers - 1):
            mlp = nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim)
            )
            self.convs.append(GINEConv(mlp, edge_dim=num_edge_features))
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(hidden_dim, 1)
        )

    def forward(self, x, edge_index, batch, edge_attr=None):
        x = x.float()
        ea = edge_attr.float() if edge_attr is not None else \
             torch.zeros(edge_index.shape[1], 3, device=x.device)
        for conv in self.convs:
            x = F.relu(conv(x, edge_index, ea))
            x = F.dropout(x, p=self.dropout, training=self.training)
        return self.head(global_add_pool(x, batch)).view(-1)

NUM_EDGE_FEATURES = dataset[0].edge_attr.shape[1]  # 3
gine_model = GINEClassifier(NUM_FEATURES, NUM_EDGE_FEATURES, HIDDEN_DIM, NUM_LAYERS, DROPOUT).to(DEVICE)
n_params = sum(p.numel() for p in gine_model.parameters() if p.requires_grad)
print(f'GINE parameters: {n_params:,}')
print(f'Edge features used: {NUM_EDGE_FEATURES} (bond type, stereo, in-ring)')

### Architecture Visualisation

We use **torchview** to render a computation graph for one forward pass, and a **matplotlib** diagram showing the layer-by-layer data flow — useful when torchview is unavailable (e.g. on Colab without Graphviz).

In [ ]:
# ── Architecture Visualisation ───────────────────────────────────────────────
# Requires: pip install torchview graphviz
# torchview traces a real forward pass so it works with PyG's variable-size graphs.

import subprocess, sys

# ── Part A: torchview (Graphviz-based computation graph) ─────────────────────
try:
    from torchview import draw_graph
    HAS_TORCHVIEW = True
except ImportError:
    print('torchview not found — installing...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torchview'], check=True)
    try:
        from torchview import draw_graph
        HAS_TORCHVIEW = True
    except ImportError:
        HAS_TORCHVIEW = False
        print('torchview unavailable — skipping Graphviz graph')

if HAS_TORCHVIEW:
    # Build a minimal sample batch from the first training molecule
    _sample = Batch.from_data_list([train_data[0]]).to('cpu')
    _vis_model = GINClassifier(NUM_FEATURES, HIDDEN_DIM, NUM_LAYERS, DROPOUT)

    # torchview needs concrete inputs — pass x, edge_index, batch as a tuple
    graph_viz = draw_graph(
        _vis_model,
        input_data=(_sample.x.float(), _sample.edge_index, _sample.batch),
        graph_name='GIN Classifier',
        roll=True,           # unroll recurrent-style loops
        depth=4,             # how deep to expand nested modules
        device='cpu',
        hide_module_functions=False,
    )
    print('torchview graph rendered — calling .visual_graph to display:')
    graph_viz.visual_graph   # renders inline in Jupyter

# ── Part B: matplotlib layer diagram (always shown) ──────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch

fig, ax = plt.subplots(figsize=(14, 5))
ax.set_xlim(0, 14)
ax.set_ylim(0, 5)
ax.axis('off')
ax.set_facecolor('#f8fafc')
fig.patch.set_facecolor('#f8fafc')

# Layer definitions: (x_center, label_top, label_bot, color, width)
layers = [
    (0.9,  'Input',           f'[N × {NUM_FEATURES}]\natom features',   '#e0f2fe', 1.4),
    (2.7,  'GINConv #1',      f'MLP({NUM_FEATURES}→{HIDDEN_DIM})\n+ ReLU + Dropout', '#dbeafe', 1.6),
    (4.6,  'GINConv #2–4',    f'MLP({HIDDEN_DIM}→{HIDDEN_DIM})\n× 3 layers',        '#dbeafe', 1.6),
    (6.5,  'Global\nSum Pool', f'[graphs × {HIDDEN_DIM}]\nΣ over atoms',             '#dcfce7', 1.4),
    (8.3,  'Linear\n+ ReLU',  f'{HIDDEN_DIM} → {HIDDEN_DIM}',                       '#fef9c3', 1.4),
    (10.1, 'Dropout',         f'p = {DROPOUT}\n(train only)',                        '#fce7f3', 1.2),
    (11.7, 'Linear',          f'{HIDDEN_DIM} → 1\n(logit)',                          '#fef3c7', 1.2),
    (13.2, 'Sigmoid',         'Output\nP(HIV active)',                               '#d1fae5', 1.2),
]

box_h = 1.7
box_y = 1.65

prev_right = None
for (xc, top, bot, color, w) in layers:
    x0 = xc - w/2
    rect = mpatches.FancyBboxPatch(
        (x0, box_y), w, box_h,
        boxstyle='round,pad=0.08',
        facecolor=color, edgecolor='#94a3b8', linewidth=1.2
    )
    ax.add_patch(rect)
    ax.text(xc, box_y + box_h*0.68, top, ha='center', va='center',
            fontsize=8.5, fontweight='bold', color='#1e293b')
    ax.text(xc, box_y + box_h*0.28, bot, ha='center', va='center',
            fontsize=7,   color='#475569', linespacing=1.5)
    if prev_right is not None:
        ax.annotate('', xy=(x0, box_y + box_h/2),
                    xytext=(prev_right, box_y + box_h/2),
                    arrowprops=dict(arrowstyle='->', color='#64748b', lw=1.5))
    prev_right = x0 + w

# Brace annotation for repeated GINConv
ax.annotate('', xy=(5.5, box_y - 0.35), xytext=(1.9, box_y - 0.35),
            arrowprops=dict(arrowstyle='<->', color='#3b82f6', lw=1.3))
ax.text(3.7, box_y - 0.62, f'Message passing ({NUM_LAYERS} hops — each atom sees {NUM_LAYERS}-hop neighbourhood)',
        ha='center', fontsize=7.5, color='#3b82f6', style='italic')

# MLP head brace
ax.annotate('', xy=(12.3, box_y - 0.35), xytext=(7.6, box_y - 0.35),
            arrowprops=dict(arrowstyle='<->', color='#f59e0b', lw=1.3))
ax.text(9.9, box_y - 0.62, 'MLP classification head',
        ha='center', fontsize=7.5, color='#b45309', style='italic')

ax.set_title(
    f'GINClassifier — {NUM_LAYERS}-layer Graph Isomorphism Network   '
    f'(hidden_dim={HIDDEN_DIM}, dropout={DROPOUT})',
    fontsize=11, fontweight='bold', color='#0f172a', pad=12
)

plt.tight_layout()
plt.savefig('/tmp/gin_architecture.png', dpi=150, bbox_inches='tight')
plt.show()
print('Architecture diagram saved to /tmp/gin_architecture.png')


## 6. Training

### Handling Class Imbalance (~3.5% positive)

Without correction, a model that always predicts "inactive" gets 96.5% accuracy but ROC-AUC ≈ 0.5 (useless).  
Fix: `BCEWithLogitsLoss(pos_weight=w)` where **w = n_neg / n_pos ≈ 25.7**.  
Each positive sample is penalised 25.7× more than a negative, forcing the model to care about the minority class.

In [ ]:
# Compute pos_weight from training set
train_labels = torch.cat([data.y for data in train_data]).view(-1).float()
train_labels = train_labels[~torch.isnan(train_labels)]
n_pos_train = float((train_labels == 1).sum())
n_neg_train = float((train_labels == 0).sum())
pos_weight_val = n_neg_train / n_pos_train
print(f'Training set — Positive: {int(n_pos_train):,}, Negative: {int(n_neg_train):,}')
print(f'pos_weight = {pos_weight_val:.2f}')

In [ ]:
def run_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, total_count = 0.0, 0
    for batch in loader:
        batch = batch.to(DEVICE)
        y = batch.y.view(-1).float()
        mask = ~torch.isnan(y)
        if mask.sum() == 0:
            continue
        optimizer.zero_grad()
        logits = model(batch.x, batch.edge_index, batch.batch)
        loss = criterion(logits[mask], y[mask])
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * int(mask.sum())
        total_count += int(mask.sum())
    return total_loss / max(total_count, 1)


@torch.no_grad()
def evaluate_loader(model, loader):
    model.eval()
    y_true, y_score = [], []
    for batch in loader:
        batch = batch.to(DEVICE)
        y = batch.y.view(-1).float()
        mask = ~torch.isnan(y)
        if mask.sum() == 0:
            continue
        logits = model(batch.x, batch.edge_index, batch.batch)
        probs = torch.sigmoid(logits)
        y_true.append(y[mask].cpu().numpy())
        y_score.append(probs[mask].cpu().numpy())
    if not y_true:
        return {'roc_auc': float('nan'), 'pr_auc': float('nan')}
    yt = np.concatenate(y_true)
    ys = np.concatenate(y_score)
    try:
        roc = roc_auc_score(yt, ys)
    except Exception:
        roc = float('nan')
    try:
        pr = average_precision_score(yt, ys)
    except Exception:
        pr = float('nan')
    return {'roc_auc': roc, 'pr_auc': pr}

In [ ]:
# ── Training config ───────────────────────────────────────────────────────────
EPOCHS    = 30   # increase to 50+ for best results
LR        = 1e-3
WD        = 1e-5
PATIENCE  = 10

model     = GINClassifier(NUM_FEATURES, HIDDEN_DIM, NUM_LAYERS, DROPOUT).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WD)
criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([pos_weight_val], device=DEVICE)
)

# ── Training loop ─────────────────────────────────────────────────────────────
history = []
best_val_auc = 0.0
patience_counter = 0
best_ckpt_path = Path('/tmp/best_gnn.pt')

for epoch in range(1, EPOCHS + 1):
    train_loss = run_epoch(model, train_loader, optimizer, criterion)
    val_metrics = evaluate_loader(model, valid_loader)
    val_auc = val_metrics['roc_auc']
    
    history.append({
        'epoch': epoch, 'train_loss': train_loss,
        'val_roc_auc': val_auc, 'val_pr_auc': val_metrics['pr_auc']
    })
    
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        patience_counter = 0
        torch.save({
            'model_state': model.state_dict(),
            'epoch': epoch,
            'val_roc_auc': val_auc,
            'num_node_features': NUM_FEATURES,
            'args': {'hidden_dim': HIDDEN_DIM, 'num_layers': NUM_LAYERS, 'dropout': DROPOUT},
        }, best_ckpt_path)
        marker = ' ✓ best'
    else:
        patience_counter += 1
        marker = f' (patience {patience_counter}/{PATIENCE})'
    
    print(f'Epoch {epoch:3d}/{EPOCHS} | Loss {train_loss:.4f} | Val ROC-AUC {val_auc:.4f}{marker}')
    
    if patience_counter >= PATIENCE:
        print(f'Early stopping at epoch {epoch}')
        break

print(f'\nBest validation ROC-AUC: {best_val_auc:.4f}')

## 6b. Train All Three Models & Build Ensemble

We train GIN, GCN, and GINE independently with the same hyperparameters and pos_weight. At inference we **average their sigmoid probabilities** (equal weights).

**Why ensembles work:**
- GIN: sum aggregation — preserves neighbourhood size
- GCN: mean aggregation + symmetric normalisation — different inductive bias
- GINE: sum aggregation + edge features — sees bond-level chemistry the others miss

These three make *different errors* → averaging reduces variance → better ROC-AUC.

> OGB leaderboard **#1** (0.8476) and **#2** (0.8475) are both multi-model ensembles.

In [ ]:
# ── Helper: train one model, return best checkpoint ──────────────────────────
def train_model(model, model_name, train_loader, valid_loader, test_loader,
                criterion, epochs=EPOCHS, patience=PATIENCE, lr=LR, wd=WD):
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    best_val, best_state, best_test = 0.0, None, 0.0
    wait = 0
    hist = []
    for epoch in range(1, epochs + 1):
        # Train
        model.train()
        total_loss, total_n = 0.0, 0
        for batch in train_loader:
            batch = batch.to(DEVICE)
            y = batch.y.view(-1).float()
            mask = ~torch.isnan(y)
            if mask.sum() == 0: continue
            opt.zero_grad()
            logits = model(batch.x, batch.edge_index, batch.batch,
                           **({'edge_attr': batch.edge_attr} if model_name == 'gine' else {}))
            loss = criterion(logits[mask], y[mask])
            loss.backward()
            opt.step()
            total_loss += loss.item() * int(mask.sum())
            total_n += int(mask.sum())
        # Evaluate
        def _eval(loader):
            model.eval()
            yt, ys = [], []
            with torch.no_grad():
                for b in loader:
                    b = b.to(DEVICE)
                    y = b.y.view(-1).float()
                    mask = ~torch.isnan(y)
                    if mask.sum() == 0: continue
                    lg = model(b.x, b.edge_index, b.batch,
                               **({'edge_attr': b.edge_attr} if model_name == 'gine' else {}))
                    yt.append(y[mask].cpu().numpy())
                    ys.append(torch.sigmoid(lg)[mask].cpu().numpy())
            if not yt: return 0.0
            return roc_auc_score(np.concatenate(yt), np.concatenate(ys))
        val_auc  = _eval(valid_loader)
        test_auc = _eval(test_loader)
        hist.append({'epoch': epoch, 'val_auc': val_auc, 'test_auc': test_auc})
        if val_auc > best_val:
            best_val   = val_auc
            best_test  = test_auc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            wait = 0
            marker = ' ✓'
        else:
            wait += 1
            marker = ''
        if epoch % 5 == 0 or marker:
            print(f'  [{model_name.upper()}] Ep {epoch:3d} | Val {val_auc:.4f} | Test {test_auc:.4f}{marker}')
        if wait >= patience: break
    model.load_state_dict(best_state)
    return model, best_val, best_test, hist

print('Training 3 models independently...')
print('(Reduce EPOCHS to 10–15 for a quick test; use 30+ for best results)\n')

gin_model  = GINClassifier(NUM_FEATURES, HIDDEN_DIM, NUM_LAYERS, DROPOUT).to(DEVICE)
gcn_model  = GCNClassifier(NUM_FEATURES, HIDDEN_DIM, NUM_LAYERS, DROPOUT).to(DEVICE)
gine_model = GINEClassifier(NUM_FEATURES, NUM_EDGE_FEATURES, HIDDEN_DIM, NUM_LAYERS, DROPOUT).to(DEVICE)

gin_model,  gin_val,  gin_test,  gin_hist  = train_model(gin_model,  'gin',  train_loader, valid_loader, test_loader, criterion)
gcn_model,  gcn_val,  gcn_test,  gcn_hist  = train_model(gcn_model,  'gcn',  train_loader, valid_loader, test_loader, criterion)
gine_model, gine_val, gine_test, gine_hist = train_model(gine_model, 'gine', train_loader, valid_loader, test_loader, criterion)

print(f'\nIndividual results:')
print(f'  GIN   — Val: {gin_val:.4f}  Test: {gin_test:.4f}')
print(f'  GCN   — Val: {gcn_val:.4f}  Test: {gcn_test:.4f}')
print(f'  GINE  — Val: {gine_val:.4f}  Test: {gine_test:.4f}')

In [ ]:
# ── Ensemble evaluation ───────────────────────────────────────────────────────
@torch.no_grad()
def ensemble_predict(loader):
    """Average sigmoid probabilities from GIN, GCN, and GINE."""
    for m in [gin_model, gcn_model, gine_model]:
        m.eval()
    y_true, y_scores = [], [[], [], []]  # [model_idx][batch]
    for batch in loader:
        batch = batch.to(DEVICE)
        y = batch.y.view(-1).float()
        mask = ~torch.isnan(y)
        if mask.sum() == 0: continue
        y_true.append(y[mask].cpu().numpy())
        for i, (m, name) in enumerate([(gin_model, 'gin'), (gcn_model, 'gcn'), (gine_model, 'gine')]):
            kw = {'edge_attr': batch.edge_attr} if name == 'gine' else {}
            lg = m(batch.x, batch.edge_index, batch.batch, **kw)
            y_scores[i].append(torch.sigmoid(lg)[mask].cpu().numpy())
    yt = np.concatenate(y_true)
    probs = [np.concatenate(s) for s in y_scores]
    avg_prob = np.mean(probs, axis=0)
    return yt, avg_prob, probs

yt, ensemble_prob, individual_probs = ensemble_predict(test_loader)
ensemble_auc = roc_auc_score(yt, ensemble_prob)

print(f'\n=== Final Test ROC-AUC Comparison ===')
print(f'  GIN alone   : {gin_test:.4f}')
print(f'  GCN alone   : {gcn_test:.4f}')
print(f'  GINE alone  : {gine_test:.4f}')
print(f'  ─────────────────────────────')
print(f'  ENSEMBLE    : {ensemble_auc:.4f}  ← averaged predictions')

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Bar comparison
models_names  = ['GIN', 'GCN', 'GINE', 'Ensemble']
models_scores = [gin_test, gcn_test, gine_test, ensemble_auc]
colors = ['#3b82f6', '#6b7280', '#10b981', '#ef4444']
bars = axes[0].bar(models_names, models_scores, color=colors, edgecolor='white', linewidth=1.2)
axes[0].set_ylim(0.5, min(1.0, max(models_scores) + 0.08))
axes[0].axhline(0.771, color='black', linestyle='--', alpha=0.5, label='OGB GIN baseline (0.771)')
axes[0].set_ylabel('Test ROC-AUC')
axes[0].set_title('Individual vs Ensemble')
for bar, score in zip(bars, models_scores):
    axes[0].text(bar.get_x() + bar.get_width()/2, score + 0.003,
                 f'{score:.4f}', ha='center', fontweight='bold', fontsize=9)
axes[0].legend(fontsize=8)

# ROC curves
from sklearn.metrics import RocCurveDisplay
for probs, name, color in zip(individual_probs, ['GIN', 'GCN', 'GINE'],
                               ['#3b82f6', '#6b7280', '#10b981']):
    auc = roc_auc_score(yt, probs)
    RocCurveDisplay.from_predictions(yt, probs, ax=axes[1],
                                      name=f'{name} ({auc:.3f})', color=color)
RocCurveDisplay.from_predictions(yt, ensemble_prob, ax=axes[1],
                                  name=f'Ensemble ({ensemble_auc:.3f})',
                                  color='#ef4444', lw=2.5)
axes[1].plot([0,1],[0,1],'k--', alpha=0.3, label='Random (0.500)')
axes[1].set_title('ROC Curves — Test Set')
axes[1].legend(fontsize=8)

plt.suptitle('GNN Ensemble Results', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Ensemble single-molecule prediction ──────────────────────────────────────
@torch.no_grad()
def ensemble_predict_smiles(smiles: str) -> dict:
    """Predict HIV inhibition probability using the GIN+GCN+GINE ensemble."""
    from torch_geometric.data import Batch as PyGBatch
    graph = smiles_to_pyg(smiles)
    batch = PyGBatch.from_data_list([graph]).to(DEVICE)
    probs = {}
    for m, name in [(gin_model,'GIN'), (gcn_model,'GCN'), (gine_model,'GINE')]:
        m.eval()
        kw = {'edge_attr': batch.edge_attr} if name == 'GINE' else {}
        lg = m(batch.x, batch.edge_index, batch.batch, **kw)
        probs[name] = torch.sigmoid(lg).item()
    ensemble_prob = np.mean(list(probs.values()))
    return {'ensemble': ensemble_prob, 'individual': probs,
            'prediction': 'ACTIVE' if ensemble_prob >= 0.5 else 'INACTIVE'}

# Test on known HIV drugs vs non-drugs
test_molecules = {
    'AZT (NRTI, HIV drug)':       'CC1=CN(C(=O)NC1=O)[C@@H]2C[C@@H](N=[N+]=[N-])[C@H](CO)O2',
    'Efavirenz (NNRTI, HIV drug)': 'FC(F)(F)C1(CC#C)OC2=NC(=NC2=C1)Cl',
    'Aspirin (non-HIV drug)':      'CC(=O)OC1=CC=CC=C1C(=O)O',
    'Caffeine (stimulant)':        'CN1C=NC2=C1C(=O)N(C(=O)N2C)C',
}

print(f'{"Molecule":<35} {"Ensemble":>8}  {"GIN":>6}  {"GCN":>6}  {"GINE":>6}  Prediction')
print('─' * 85)
for name, smi in test_molecules.items():
    r = ensemble_predict_smiles(smi)
    print(f'{name:<35} {r["ensemble"]:>8.3f}  '
          f'{r["individual"]["GIN"]:>6.3f}  '
          f'{r["individual"]["GCN"]:>6.3f}  '
          f'{r["individual"]["GINE"]:>6.3f}  '
          f'{r["prediction"]}')

## 7. Evaluation & Results

In [ ]:
# Load best checkpoint and evaluate on test set
ckpt = torch.load(best_ckpt_path, map_location=DEVICE)
model.load_state_dict(ckpt['model_state'])
model.eval()

test_metrics = evaluate_loader(model, test_loader)
print(f'Test ROC-AUC : {test_metrics["roc_auc"]:.4f}')
print(f'Test PR-AUC  : {test_metrics["pr_auc"]:.4f}')
print(f'\nBaselines for comparison:')
print(f'  Random baseline     : 0.500')
print(f'  Fingerprint + LR    : ~0.680')
print(f'  OGB GCN (official)  : 0.759')
print(f'  OGB GIN (official)  : 0.771')

In [ ]:
# Training curves + ROC curve
hist_df = pd.DataFrame(history)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Loss
axes[0].plot(hist_df['epoch'], hist_df['train_loss'], 'b-', label='Train Loss')
axes[0].set_title('Training Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('BCE Loss')
axes[0].legend()

# Val ROC-AUC
axes[1].plot(hist_df['epoch'], hist_df['val_roc_auc'], 'g-', label='Val ROC-AUC')
axes[1].axhline(y=best_val_auc, color='red', linestyle='--', alpha=0.5, label=f'Best={best_val_auc:.3f}')
axes[1].set_title('Validation ROC-AUC')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('ROC-AUC')
axes[1].legend()

# ROC curve on test set
y_true_all, y_score_all = [], []
with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(DEVICE)
        y = batch.y.view(-1).float()
        mask = ~torch.isnan(y)
        if mask.sum() == 0:
            continue
        probs = torch.sigmoid(model(batch.x, batch.edge_index, batch.batch))
        y_true_all.append(y[mask].cpu().numpy())
        y_score_all.append(probs[mask].cpu().numpy())

y_true_all  = np.concatenate(y_true_all)
y_score_all = np.concatenate(y_score_all)
RocCurveDisplay.from_predictions(y_true_all, y_score_all, ax=axes[2], name='GIN')
axes[2].plot([0,1],[0,1],'k--', alpha=0.4, label='Random')
axes[2].set_title('Test ROC Curve')
axes[2].legend()

plt.suptitle('GIN Training Results', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Score distribution: active vs inactive
scores_active   = y_score_all[y_true_all == 1]
scores_inactive = y_score_all[y_true_all == 0]

plt.figure(figsize=(8, 4))
plt.hist(scores_inactive, bins=50, alpha=0.6, color='#6b7280', label=f'Inactive (n={len(scores_inactive):,})', density=True)
plt.hist(scores_active,   bins=50, alpha=0.7, color='#ef4444', label=f'Active (n={len(scores_active):,})', density=True)
plt.axvline(0.5, color='black', linestyle='--', label='Threshold 0.5')
plt.xlabel('Predicted HIV inhibition probability')
plt.ylabel('Density')
plt.title('Score Distribution — Test Set')
plt.legend()
plt.tight_layout()
plt.show()
print(f'Active molecules — mean score   : {scores_active.mean():.3f}')
print(f'Inactive molecules — mean score : {scores_inactive.mean():.3f}')

## 8. Single Molecule Prediction

Predict HIV inhibition probability for any molecule given its SMILES string.

In [ ]:
@torch.no_grad()
def predict_smiles(smiles: str, model=model, device=DEVICE) -> dict:
    """Predict HIV inhibition probability from a SMILES string."""
    try:
        from rdkit import Chem
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return {'error': 'Invalid SMILES'}
    except ImportError:
        pass  # skip rdkit validation if not available
    
    graph = smiles_to_pyg(smiles)
    batch = Batch.from_data_list([graph]).to(device)
    model.eval()
    logit = model(batch.x, batch.edge_index, batch.batch)
    prob  = torch.sigmoid(logit).item()
    return {
        'smiles': smiles,
        'probability': prob,
        'prediction': 'ACTIVE (HIV inhibitor)' if prob >= 0.5 else 'INACTIVE',
        'confidence': 'high' if abs(prob - 0.5) > 0.3 else 'low',
        'atoms': graph.num_nodes,
    }

# Test known HIV drugs vs non-drugs
test_molecules = {
    # Known HIV drugs
    'AZT (NRTI, HIV drug)':        'CC1=CN(C(=O)NC1=O)[C@@H]2C[C@@H](N=[N+]=[N-])[C@H](CO)O2',
    'Efavirenz (NNRTI, HIV drug)':  'FC(F)(F)C1(CC#C)OC2=NC(=NC2=C1)Cl',
    'Raltegravir (INSTI, HIV drug)':'CC(=O)NC1=NC(=O)C(=C(N1)O)C(=O)NCC2=CC(=CC=C2)F',
    # Non-HIV drugs
    'Aspirin (anti-inflammatory)':  'CC(=O)OC1=CC=CC=C1C(=O)O',
    'Caffeine (stimulant)':         'CN1C=NC2=C1C(=O)N(C(=O)N2C)C',
    'Paracetamol (pain relief)':    'CC(=O)Nc1ccc(O)cc1',
}

print(f'{"Molecule":<35} {"Prob":>6}  Prediction')
print('─' * 70)
for name, smi in test_molecules.items():
    result = predict_smiles(smi)
    prob = result.get('probability', 0)
    pred = result.get('prediction', 'ERROR')
    bar = '█' * int(prob * 20)
    print(f'{name:<35} {prob:>6.3f}  {pred}')

## 9. Tox21 Virtual Screening

The **Tox21** dataset (8,014 compounds) was assembled by the NIH to assess toxicity of environmental chemicals and drugs. Here we repurpose it as a **virtual screening library**: we screen every Tox21 compound through our trained GNN to rank them by predicted HIV inhibition probability.

### What this is — and what it isn't
- ✅ **Is:** A computational pre-screen using structural similarity to known HIV inhibitors
- ✅ **Is:** A hypothesis generator — compounds worth wet-lab follow-up
- ❌ **Isn't:** Mechanistic prediction (we don't know if hits work via RT, integrase, capsid, etc.)
- ❌ **Isn't:** A guarantee of activity — the model only learned phenotypic correlations from the NCI assay

In [ ]:
from ogb.graphproppred import PygGraphPropPredDataset as OGBDataset

print('Loading Tox21 dataset...')
try:
    tox21 = OGBDataset(name='ogbg-moltox21')
    print(f'Tox21: {len(tox21):,} compounds, {tox21.num_tasks} toxicity tasks')
    TOX21_AVAILABLE = True
except Exception as e:
    print(f'Could not load via OGB: {e}')
    TOX21_AVAILABLE = False

In [ ]:
if not TOX21_AVAILABLE:
    # Fallback: download Tox21 SMILES directly
    import urllib.request
    TOX21_URL = 'https://raw.githubusercontent.com/deepchem/deepchem/master/datasets/tox21.csv'
    tox21_path = Path('/tmp/tox21.csv')
    if not tox21_path.exists():
        print('Downloading Tox21 CSV...')
        urllib.request.urlretrieve(TOX21_URL, tox21_path)
    tox21_df = pd.read_csv(tox21_path)
    smiles_col = 'smiles' if 'smiles' in tox21_df.columns else tox21_df.columns[-1]
    tox21_smiles = tox21_df[smiles_col].dropna().tolist()
    print(f'Tox21 CSV: {len(tox21_smiles):,} molecules')
    TOX21_AVAILABLE = True
    USE_CSV = True
else:
    USE_CSV = False

In [ ]:
@torch.no_grad()
def screen_smiles_list(smiles_list: list[str], model, device, batch_size: int = 256) -> pd.DataFrame:
    """
    Screen a list of SMILES strings through the trained GNN.
    Returns a DataFrame sorted by predicted HIV inhibition probability (descending).
    """
    model.eval()
    results = []
    skipped = 0
    
    # Build graphs
    graphs, valid_smiles = [], []
    for smi in tqdm(smiles_list, desc='Converting SMILES → graphs'):
        try:
            g = smiles_to_pyg(smi)
            graphs.append(g)
            valid_smiles.append(smi)
        except Exception:
            skipped += 1
    
    print(f'Valid: {len(graphs):,}, Skipped (invalid SMILES): {skipped}')
    
    # Batch inference
    all_probs = []
    for i in tqdm(range(0, len(graphs), batch_size), desc='Running GNN inference'):
        batch_graphs = graphs[i:i+batch_size]
        batch = Batch.from_data_list(batch_graphs).to(device)
        logits = model(batch.x, batch.edge_index, batch.batch)
        probs  = torch.sigmoid(logits).cpu().numpy()
        all_probs.extend(probs.tolist())
    
    df = pd.DataFrame({'smiles': valid_smiles, 'hiv_inhibition_prob': all_probs})
    df = df.sort_values('hiv_inhibition_prob', ascending=False).reset_index(drop=True)
    df['rank'] = df.index + 1
    return df


# Run screening
if USE_CSV:
    screen_input = tox21_smiles
else:
    # Get SMILES from OGB Tox21
    try:
        import pandas as pd
        raw = pd.read_csv(Path(tox21.root) / 'mapping' / 'mol.csv.gz')
        scol = [c for c in raw.columns if 'smiles' in c.lower()][0]
        screen_input = raw[scol].dropna().tolist()
    except Exception:
        screen_input = [smiles_to_pyg.__doc__]  # fallback

screen_df = screen_smiles_list(screen_input, model, DEVICE)
print(f'\nScreened {len(screen_df):,} compounds')
print(f'Compounds with prob > 0.5: {(screen_df.hiv_inhibition_prob > 0.5).sum()}')
print(f'Compounds with prob > 0.3: {(screen_df.hiv_inhibition_prob > 0.3).sum()}')

In [ ]:
# Top 20 candidates
threshold = 0.5
top_candidates = screen_df[screen_df['hiv_inhibition_prob'] >= threshold].head(20)

print(f'Top candidates (prob ≥ {threshold}):')
print('─' * 80)
print(f'{"Rank":>4}  {"Prob":>6}  SMILES')
print('─' * 80)
for _, row in top_candidates.iterrows():
    smi_trunc = row['smiles'][:55] + '...' if len(row['smiles']) > 55 else row['smiles']
    print(f'{int(row["rank"]):>4}  {row["hiv_inhibition_prob"]:>6.3f}  {smi_trunc}')

In [ ]:
# Visualise screening results
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Probability distribution
axes[0].hist(screen_df['hiv_inhibition_prob'], bins=60, color='#3b82f6', edgecolor='white', alpha=0.8)
axes[0].axvline(0.5, color='red', linestyle='--', linewidth=2, label='Threshold 0.5')
axes[0].axvline(0.3, color='orange', linestyle=':', linewidth=1.5, label='Threshold 0.3')
n_hits = (screen_df['hiv_inhibition_prob'] >= 0.5).sum()
axes[0].set_xlabel('Predicted HIV Inhibition Probability')
axes[0].set_ylabel('Number of Compounds')
axes[0].set_title(f'Tox21 Screening Distribution\n({len(screen_df):,} compounds, {n_hits} hits ≥ 0.5)')
axes[0].legend()

# Top 30 ranked compounds
top30 = screen_df.head(30)
colors = ['#ef4444' if p >= 0.5 else '#f59e0b' if p >= 0.3 else '#6b7280'
          for p in top30['hiv_inhibition_prob']]
axes[1].barh(range(len(top30)), top30['hiv_inhibition_prob'], color=colors)
axes[1].set_yticks(range(len(top30)))
axes[1].set_yticklabels([f'#{i+1}' for i in range(len(top30))], fontsize=8)
axes[1].axvline(0.5, color='red', linestyle='--', linewidth=1.5, label='Threshold 0.5')
axes[1].set_xlabel('HIV Inhibition Probability')
axes[1].set_title('Top 30 Tox21 Candidates')
axes[1].invert_yaxis()
red_p   = mpatches.Patch(color='#ef4444', label='High (≥0.5)')
yel_p   = mpatches.Patch(color='#f59e0b', label='Medium (0.3–0.5)')
gray_p  = mpatches.Patch(color='#6b7280', label='Low (<0.3)')
axes[1].legend(handles=[red_p, yel_p, gray_p], loc='lower right', fontsize=8)

plt.suptitle('Tox21 Virtual Screening Results', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Optional: visualise top candidate structures with RDKit
try:
    from rdkit import Chem
    from rdkit.Chem import Draw
    from IPython.display import display, Image
    import io

    top_smiles = screen_df.head(6)['smiles'].tolist()
    top_probs  = screen_df.head(6)['hiv_inhibition_prob'].tolist()

    mols   = [Chem.MolFromSmiles(s) for s in top_smiles]
    labels = [f'Rank {i+1}\nP={p:.3f}' for i, p in enumerate(top_probs)]
    valid  = [(m, l) for m, l in zip(mols, labels) if m is not None]

    if valid:
        img = Draw.MolsToGridImage(
            [v[0] for v in valid],
            molsPerRow=3,
            subImgSize=(350, 280),
            legends=[v[1] for v in valid]
        )
        img.save('/tmp/top_tox21_candidates.png')
        display(img)
        print('Top Tox21 candidates rendered above.')
except ImportError:
    print('RDKit Draw not available — skipping structure visualisation.')

In [ ]:
# Save results to CSV
output_path = Path('/tmp/tox21_screening_results.csv')
screen_df.to_csv(output_path, index=False)
print(f'Full results saved to: {output_path}')
print(f'Columns: {list(screen_df.columns)}')
print(f'\nTop 5 hits:')
print(screen_df.head(5).to_string(index=False))

## 10. Key Takeaways

### What we built
- A **Graph Isomorphism Network (GIN)** that reads molecular structure as a graph and predicts HIV inhibition probability
- Trained on 32,901 real experimental measurements from the **NCI AIDS Antiviral Screen**
- Screened all **~8,000 Tox21 compounds** in one batch inference pass

### The phenotypic data caveat

| | Phenotypic (this dataset) | Mechanistic |
|---|---|---|
| **Measures** | Does HIV replication go down in T-cells? | Does molecule bind RT / integrase / protease? |
| **Tells you** | Whether it works | How it works |
| **Model learns** | Structure → cell-level outcome correlation | Structure → target affinity |
| **Downstream** | Identify candidates for lab testing | Design structure-activity relationships |

A high model score does **not** tell you whether a hit works via RT inhibition, integrase blocking, capsid disruption, or something entirely novel. That requires follow-up biochemical assays.

### Performance summary

| Model | Test ROC-AUC |
|---|---|
| Random | 0.500 |
| Fingerprint + Logistic Regression | ~0.680 |
| OGB GCN baseline | 0.759 |
| OGB GIN baseline | 0.771 |
| **This notebook (GIN + pos_weight)** | **~0.750** |

### How to improve
1. **Use edge features in GIN layers** — bond type/aromaticity in `edge_attr` is currently unused
2. **Pre-trained GNN** — Hu et al. 2020 strategy: pre-train on 2M molecules, fine-tune on molhiv (+3%)
3. **Graph Transformer (GPS)** — achieves ~0.785 on molhiv
4. **3D conformers** — add geometry via RDKit ETKDG + EGNN/SchNet
5. **Calibration** — Platt scaling to make `p=0.3` mean actual 30% probability